In [1]:
# ===== 1. Imports =====
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib

# ===== 2. Load data =====
df = pd.read_csv("loan_approval_data.csv")

# ===== 3. Handle missing values =====
categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

# ===== 4. Drop ID column =====
df = df.drop("Applicant_ID", axis=1)

# ===== 5. Encoding — SEPARATE encoders (matches app.py) =====
education_encoder = LabelEncoder()
df["Education_Level"] = education_encoder.fit_transform(df["Education_Level"])

target_encoder = LabelEncoder()
df["Loan_Approved"] = target_encoder.fit_transform(df["Loan_Approved"])

cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df.index)

df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

# ===== 6. Feature engineering =====
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

X = df.drop(columns=["Loan_Approved", "Credit_Score", "DTI_Ratio"])
y = df["Loan_Approved"]

# ===== 7. Train-test split =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===== 8. Scaling =====
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ===== 9. Train Naive Bayes =====
naive_model = GaussianNB()
naive_model.fit(X_train_scaled, y_train)

y_pred = naive_model.predict(X_test_scaled)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# ===== 10. Save files — EXACT names app.py expects =====
joblib.dump(naive_model, "creditwise_naive_bayes_v1.pkl")
joblib.dump(scaler, "creditwise_scaler_v1.pkl")
joblib.dump(ohe, "creditwise_encoder_v1.pkl")
joblib.dump(education_encoder, "creditwise_education_encoder_v1.pkl")
joblib.dump(target_encoder, "creditwise_target_encoder_v1.pkl")
joblib.dump(list(X.columns), "creditwise_feature_columns_v1.pkl")

print("All 6 files saved.")

Precision: 0.7833333333333333
Recall: 0.7704918032786885
F1 score: 0.7768595041322314
Accuracy: 0.865
All 6 files saved.


In [2]:
import os
for f in os.listdir():
    if f.endswith(".pkl"):
        print(f)

creditwise_education_encoder_v1.pkl
creditwise_encoder_v1.pkl
creditwise_feature_columns_v1.pkl
creditwise_label_encoder_v1.pkl
creditwise_naive_bayes_v1.pkl
creditwise_scaler_v1.pkl
creditwise_target_encoder_v1.pkl
